[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nrao/astrohack/blob/v0.10.1/docs/locit_tutorial.ipynb)

![astrohack](../_media/astrohack_logo.png)

# Antenna position correction tutorial

`extract_locit` and `locit` are utilities designed to help determine antenna position shifts after antenna relocation.
To do so they rely on a phase gain calibration table created by `CASA` from antenna pointing data.
The process in `CASA` consists of:
1. `split` out the actual pointing data from the original pointing measurement set (MS), it might contain data taken while the antennas are still slewing.
2. `fringe_fit` the MS using a single source with no delay rates, this is done to flatten a spectral window.
3. `apply_cal` the fringe_fit solution.
4. Channel average the MS using `split`.
5. Compute an average phase gain solution for each source using `gaincal`

To simplify the process in `CASA` a script is distributed within `astrohack` that the user can simply fill in the parameters for the data reduction and then run it within `CASA`.


In [ ]:
import os

try:
    import astrohack

    print("AstroHACK version", astrohack.__version__, "already installed.")
except ImportError as e:
    print(e)
    print("Installing AstroHACK")

    os.system("pip install astrohack")

    import astrohack

    print("astrohack version", astrohack.__version__, " installed.")

## Download Tutorial data

In [ ]:
# The Cal table used here is a placeholder, there should be a better dataset to be used with the tutorial
import toolviper

toolviper.utils.data.download(file="locit-input-pha.cal", folder="data")

## Position and locit Data File API

As part of the `astroHACK` API a set of functions to allow users to easily open on disk locit and position files has been provided. Each function takes an `astroHACK` locit or position file name as an argument and returns an object related to the given file type. Each object allows the user to access data via dictionary keys with values consisting of the relevant dataset. Each object also provides a `summary()` helper function to list available keys for each file. An example call for each file type is show below and the API documentation for all data-io functions can be found [here](https://astrohack.readthedocs.io/en/latest/_api/autoapi/astrohack/dio/index.html).

```python
from astrohack import open_locit
from astrohack import open_position

locit_data = open_locit(file='./data/locit-input-pha.locit.zarr')
position_data = open_position(file='./data/locit-input-pha.position.zarr')
```

## Setup Dask Local Cluster

The local Dask client handles scheduling and worker managment for the parallelization. The user has the option of choosing the number of cores and memory allocations for each worker howerver, we recommend a minimum of 1Gb per core with standard settings.


A significant amount of information related to the client and scheduling can be found using the [Dask Dashboard](https://docs.dask.org/en/stable/dashboard.html). This is a built-in dashboard native to Dask and allows the user to monitor the workers during processing. This is especially useful for profilling. For those that are interested in working soley within Jupyterlab a dashboard extension is available for [Jupyterlab](https://github.com/dask/dask-labextension#dask-jupyterlab-extension).

![dashboard](../_media/dashboard.png)

In [ ]:
from toolviper.dask.client import local_client

parallel = False

if parallel:
    client = local_client(cores=4, memory_limit="1GB")
    print(client)
else:
    client = None


## Extract locit

The first step in determining the antenna position corrections is to extract the data from the phase gains calibration table and store it in a convenient format for further processing.

In the calibration table the data is organized by time, but we want organized by antenna → DDI → time for simplicity of processing in `locit`.

Also, the data in the calibration table may contain more than one reference antenna, which would scramble the results obtained by `locit`, hence we throw away data that has a different reference antenna than the main reference antenna in `extract_locit`

In [ ]:
cal_table = "./data/locit-input-pha.cal"
locit_name = "./data/locit-input-pha.locit.zarr"
position_name = "./data/locit-input-pha.position.zarr"

In [ ]:
%%time
from astrohack import extract_locit

locit_mds = extract_locit(
    cal_table,  # The calibration table containing the phase gains
    locit_name=locit_name,  # The name for the created locit file
    ant="all",  # Antenna selection, None means 'All'
    ddi="all",  # DDI selection, None means 'ALL'
    overwrite=True,
)

`extract_locit` creates a file that is called a locit file. This file contains the phase gains for each antenna but also contains two important tables, the source and antenna tables.

`extract_locit` also returns the opened locit file as a `locit_mds` object. The first step in interacting with the `locit_mds` object is calling its summary

In [ ]:
locit_mds.summary()

From the summary, we can see that the locit file contains 26 antennas and 2 DDIs per antenna, as well as 4 different methods related to the visualization of the source and antenna tables. To inspect the data contained in a DDI for an antenna, we simply access the dictionary keys as so,

In [ ]:
locit_mds["ant_ea06"]["ddi_0"]

### Inspecting the Sources in the dataset

When trying to determine the antenna position correction, we are always interested in knowing the distribution in the sky of the sources used in the pointing observation. The antenna position corrections in X and Y are affected by the hour-angle coverage of the observations, while the Z position correction is affected by the declination coverage of the observations.

First we will print the source table, and second we will plot the sources on a simplified sky plot for easier visualization.

In [ ]:
locit_mds.print_source_table()

In [ ]:
locit_plot_folder = "locit_mds_plots"

locit_mds.plot_source_positions(
    locit_plot_folder,  # destination for the plot
    labels=True,  # Display source labels on plot
    precessed=False,  # Plot FK5 (J2000) coordinates instead of precessed coordinates
    display=True,
)

### Inspecting the array configuration in the dataset

Another important piece of information when determining antenna position corrections is which antennas are present in the observations and where are they located in the array. We have introduced two methods to display this information, the first, `print_array_configuration`, displays all the antennas for the array, accompanied by their positions if they are present in the dataset. The second method, `plot_array_configuration`, plots the positions of the antennas in the dataset; antennas not present are simply skipped.

In [ ]:
locit_mds.print_array_configuration(
    relative=False
)  # antenna positions printed are relative to the array center

In [ ]:
locit_mds.plot_array_configuration(
    locit_plot_folder,  # Folder in which to save the plot
    stations=True,  # Toggle to display the station name alongside the antenna name
    zoff=False,  # Toggle to display the antenna elevation offset by its name
    unit="km",  # Length unit for the plot
    box_size=5,  # Size of the box for the inner array in the unit specified in unit
    display=True,
)

## Locit

After we have inspected the `locit_mds` object we can now use `locit` to obtain antenna position corrections. In this dataset a single antenna, ea06 has been moved, and hence we could skip the other antennas and get position corrections for only it and the reference antenna, ea28. 
But here we will be getting corrections for all antennas as this can help point out systematic errors with the dataset, such as choosing a bad reference antenna.
We include the reference antenna in the fit as a sanity check, the position corrections for the reference antenna, as well as the fixed delay and delay rate are by construction, 0, if they aren't there is something wrong with the code.

In [ ]:
%%time
from astrohack import locit

position_mds = locit(
    locit_name,
    position_name=position_name,  # Name of the position file to be created by locit
    elevation_limit=10.0,  # Elevation under which no sources are considered
    polarization="both",  # Combine both R and L polarization phase gains for increased SNR
    fit_engine="scipy",  # Fit data using scipy
    fit_kterm=False,
    fit_delay_rate=True,  # Fit delay rate
    ant="all",  # Select all antennas
    ddi="all",  # Select all DDIs
    combine_ddis="simple",  # Combine delays from all DDIs to obtain a single solution with increased SNR
    parallel=parallel,  # Do fitting in parallel
    overwrite=True,  # Overwrite previously created position file
)

`locit` creates a file that is called a position file. This file contains the delays, and the fitted delay model for each antenna

`locit` also returns the opened position file as a `position_mds` object. The first step in interacting with the `position_mds` object is calling its summary

In [ ]:
position_mds.summary()

From the summary we can see that the position file contains simply 2 antennas and no DDIs, as well as 4 different methods:
- `export_fit_results` exports the antenna position corrections to an ascii file.
- `plot_sky_coverage` plots the sky coverage for an antenna and DDI (if present).
- `plot_delays` plots the measured delays as a function of time, hour angle, declination and elevation,
- `plot_position_corrections` Plots the position corrections on an array plot, making it easier to identify systematics

To inspect the data contained in the position file for an antenna we can then simply do:

In [ ]:
position_mds["ant_ea06"]

The following plot of the sky coverage of the sources for antenna ea06, gives us an idea of how good our results can be. From it we see that basically all possible hour-angles and declinations are covered, which implies that the position correction determinations are as good as they can be given the observing conditions are good and stable enough.

Weather may complicate this measurement by introducing anisotropic and time dependant delays, limiting the methods accuracy.

In [ ]:
position_plot_folder = "position_mds_exports"

position_mds.plot_sky_coverage(
    position_plot_folder,  # Folder to contain plot
    ant="ea06",  # Plot only antenna ea06
    ddi="all",  # DDI selection irrelevant because we are combining DDIs
    time_unit="hour",  # Unit for observation duration
    angle_unit="deg",  # Unit for sky coordinates
    display=True,
    parallel=parallel
)

Below we export the fit results to an ascii file and display it for analysis.
In it we can see that the results for the reference antenna are all 0 and the delay RMS is very small, which is indeed what is expected.

In [ ]:
position_export_folder = "position_mds_exports"

position_mds.export_locit_fit_results(
    position_export_folder,  # Folder to contain antenna position corrections file
    ant="all",  # See results for all antennas
    position_unit="mm",  # Unit for the position corrections
    delay_unit="nsec",  # Unit for delays
    time_unit="hour",  # Unit for delay rate denominator
)

Now we plot the delays and the delay model that was fitted with `locit`. From this plot we can see that model delays agree very well with the observed delays leading to a good confidence in the position corrections derived with `locit`.

In [ ]:
position_mds.plot_delays(
    position_plot_folder,  # Folder to contain plot
    ant="ea06",  # Plot only antenna ea06
    ddi="all",  # DDI selection irrelevant because we are combining DDIs
    time_unit="hour",  # Unit for observation duration
    angle_unit="deg",  # Unit for sky coordinates
    delay_unit="nsec",  # Unit for delays
    plot_model=True,  # Plot fitted delay model
    display=True,
    parallel=parallel
)

One extra way to check for systematic errors in antenna position determinations is to plot the corrections for the whole array.
If all the corrections point the same way this might be an indication that the chosen reference_antenna has an error in its position.

In [ ]:
position_mds.plot_position_corrections(
    position_plot_folder,  # Folder to contain plot
    unit="km",  # Unit for the x and Y axes
    box_size=5,  # Size for the box containing the inner array
    scaling=250,  # scaling to be applied to corrections
    display=True,
)

Finally, when we are certain that our results are good we can then export the fit results to an ASCII file that is formatted for input by the VLA parminator software.

In [ ]:
position_mds.export_results_to_parminator(
    "ant_pos_cor_24-10-14.PAR",  # name of the output parminator file
    ant=["ea06", "ea01", "ea05"],  # Selected moved antennas
    correction_threshold=0.001,  # Threshold for valid corrections in meters (i.e. minimum value for correction to appear in parminator file)
)

print("Parminator file contents:\n")
for line in open("ant_pos_cor_24-10-14.PAR"):
    print(line[:-1])

In [ ]:
if parallel:
    client.close()